In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
display(df)

In [0]:
df = spark.read.table("uber.bronze.bulk_rides")
rides_schema = df.schema
display(rides_schema)

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
df_parsed = df.withColumn("parsed_rides", from_json(col("rides"), rides_schema)).select("parsed_rides.*")
display(df_parsed)

In [0]:
pip install Jinja2

In [0]:
jinja_config = [
    {
        "table": "uber.bronze.stg_rides stg",
        "alias" : "stg",
        "select": "stg.*",
        "where": ""
    },
    {
        "table": "uber.bronze.map_vehicle_types typ",
        "alias" : "typ",
        "select": "upper(typ.vehicle_type),description, typ.base_rate, typ.per_mile",
        "on": "typ.vehicle_type_id = stg.vehicle_type_id",
        "where": ""
    },
    {
        "table": "uber.bronze.map_vehicle_makes mk",
        "alias" : "mk",
        "select": "upper(mk.vehicle_make)",
        "on": "mk.vehicle_make_id = stg.vehicle_make_id",
        "where": ""
    }
]

In [0]:
from jinja2 import Template

jinja_str = """
select {% for config in jinja_config %}
    {{ config.select }}
    {% if not loop.last %},{% endif %}
    {% endfor %}
FROM
    {% for config in jinja_config %} 
        {% if loop.first %}
            {{ config.table }}
        {% else %}
            Left join {{ config.table }} on {{ config.on }}
        {% endif %}
    {% endfor %}

    {% for config in jinja_config %}
        {% if loop.first %}
            {% if config.where != "" %}
                WHERE
            {% endif%}
        {% endif %}
        {{ config.where }}
        {% if not loop.last %}
            {% if config.where != "" %}
                AND
            {% endif%}
        {% endif %}
    {% endfor %}
    
"""

template = Template(jinja_str)
rendered_template = template.render(jinja_config = jinja_config)
print(rendered_template)

In [0]:
display(spark.sql(rendered_template))

In [0]:
%sql
select * from uber.bronze.map_vehicle_types;

In [0]:
%sql
drop table uber.bronze.stg_rides